In [642]:
import torch

In [643]:
torch.zeros(4, 8, 16).dim()

3

In [644]:
import math

import torch
from einops import einsum


class Linear(torch.nn.Module):
    weights: torch.Tensor

    def __init__(self, in_features: int, out_features: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.weights = torch.nn.parameter.Parameter(torch.empty(out_features, in_features, device=device, dtype=dtype))
        std = math.sqrt(2 / (in_features + out_features))
        torch.nn.init.trunc_normal_(tensor=self.weights, mean=0, std=std, a=-3 * std, b=3 * std)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return einsum(x, self.weights, "... in, out in -> ... out")

In [672]:
linear = Linear(1024, 1024)
linear(torch.randn(1024, 1024))

list(linear.state_dict().keys())

['weights']

In [646]:
import torch
import math
from einops import einsum

class Embedding(torch.nn.Module):

    embeddings: torch.Tensor # (num_embeddings, embedding_dim)

    def __init__(self, num_embeddings: int, embedding_dim: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.embeddings = torch.nn.parameter.Parameter(torch.empty(num_embeddings, embedding_dim, device=device, dtype=dtype))
        torch.nn.init.trunc_normal_(tensor=self.embeddings, mean=0, std=1, a=-3, b=3)

    # token_ids: torch.LongTensor (batch_size, sequence_length)
    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.embeddings[token_ids]

In [647]:
embedding = Embedding(256, 1024)
embedding(torch.randint(0, 256, (2, 5)))

tensor([[[ 0.0819, -0.5788, -0.6793,  ...,  1.6095,  0.9709, -2.3776],
         [-0.2847, -1.4779,  0.3142,  ...,  1.0670,  0.1809, -1.5505],
         [ 0.4723, -0.0907,  1.5637,  ...,  0.4456,  0.4496,  0.2777],
         [ 0.3969,  0.3203, -2.1420,  ...,  0.4255,  0.5992, -0.4830],
         [-0.8330,  2.1534,  1.0395,  ...,  1.1258,  0.7617, -0.0970]],

        [[ 0.4811,  2.0632, -0.3940,  ...,  0.1598,  0.2760, -0.8868],
         [ 0.1937,  0.4433, -0.5722,  ...,  0.1486, -1.2031,  0.6043],
         [-0.5710,  0.5475,  0.5743,  ..., -0.1203,  0.8944,  1.8348],
         [-1.6488,  1.5102,  1.2396,  ..., -0.8951,  0.0561, -0.3533],
         [ 1.5099,  0.4542,  0.5315,  ..., -2.1927, -0.4144,  0.7384]]],
       grad_fn=<IndexBackward0>)

In [648]:
import torch

class RMSNorm(torch.nn.Module):

    gain: torch.Tensor # (d_model, )
    eps: float
    
    def __init__(self, d_model: int, eps: float = 1e-5, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.gain = torch.nn.parameter.Parameter(torch.ones(d_model, device=device, dtype=dtype))
        self.eps = eps

    # x (batch_size, sequence_length, d_model)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        x = x.to(torch.float32)
        x_sqrd_mean = x.pow(2).mean(dim=-1, keepdim=True) # (batch_size, sequence_length, 1)
        rms = torch.sqrt(x_sqrd_mean + self.eps) # (batch_size, sequence_length, 1)
        # (batch_size, sequence_length, d_model) * (d_model, ) / (batch_size, sequence_length, 1)
        result = x * self.gain / rms # (batch_size, sequence_length, d_model)
        return result.to(in_dtype)


In [649]:
rms_norm = RMSNorm(1024)
rms_norm(torch.randn(1024))

tensor([-2.0366, -0.1268,  2.3768,  ..., -0.8368,  0.7291, -0.6869],
       grad_fn=<DivBackward0>)

In [650]:
import torch

class SwiGLU(torch.nn.Module):
    w_1: Linear # (d_ff, d_model)
    w_2: Linear # (d_model, d_ff)
    w_3: Linear # (d_ff, d_model)

    def __init__(self, d_model: int, d_ff: int | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        if d_ff is None:
            d_ff = round(8 * d_model / 3 / 64) * 64
        self.w_1 = Linear(d_model, d_ff, device, dtype)
        self.w_2 = Linear(d_ff, d_model, device, dtype)
        self.w_3 = Linear(d_model, d_ff, device, dtype)

    # x (batch_size, sequence_length, d_model)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_2(self._silu(self.w_1(x)) * self.w_3(x))

    def _silu(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.sigmoid(x)

In [651]:
d_model: int = 1024
swiglu = SwiGLU(d_model)
swiglu(torch.randn(256, d_model))

tensor([[ 0.2037,  0.0778, -0.3543,  ..., -0.0848,  0.2278, -0.6679],
        [-0.4096, -0.2686,  0.4487,  ...,  0.1367,  0.0561,  0.1199],
        [ 0.1882,  0.4488,  0.6400,  ...,  0.2740,  0.6063,  0.5183],
        ...,
        [ 0.6336,  0.0062, -0.1844,  ..., -0.6416, -0.1181,  0.2912],
        [ 0.3068,  0.0510,  0.2302,  ..., -0.0182,  0.0371, -0.5253],
        [ 0.1247, -0.4177,  0.5692,  ..., -0.0222, -0.9502, -0.3150]],
       grad_fn=<ViewBackward0>)

In [652]:
import torch
from einops import rearrange

class RotaryPositionalEmbedding(torch.nn.Module):

    def __init__(self, theta: float, d_k: int, max_seq_len: int, device: torch.device | None = None):
        super().__init__()
        k = torch.arange(0, d_k, 2, device=device)
        positions = torch.arange(0, max_seq_len, 1, device=device).reshape(max_seq_len, 1)
        angles = positions / theta ** (k / d_k)
        self.register_buffer('sin', torch.sin(angles), persistent=False)
        self.register_buffer('cos', torch.cos(angles), persistent=False)

    # x (batch, seq_len, d_k), token_positions (batch, seq_len)
    def forward(self, x: torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor:
        x = rearrange(x, "... seq_len (half two) -> ... seq_len half two", two=2)
        a = x[..., 0] # (..., seq, half)
        b = x[..., 1] # (..., seq, half)
        cos = self.cos[token_positions]
        sin = self.sin[token_positions]
        a_out = a * cos - b * sin
        b_out = a * sin + b * cos
        x_out = torch.stack([a_out, b_out], dim=-1) # (..., seq_len, d_k/2 2)
        x_out = rearrange(x_out, "... seq_len half two -> ... seq_len (half two)")
        return x_out

In [653]:
d_k = 4
theta = 10000
max_seq_len = 2
x = torch.tensor([[1., 0., 1., 0.], [1., 0., 1., 0.]])
rope = RotaryPositionalEmbedding(theta, d_k, max_seq_len) # theta, d_k, max_seq_len
rope.cos.shape # seq_len, d_k/2
token_positions = torch.arange(max_seq_len)
rope(x, token_positions)

tensor([[1.0000, 0.0000, 1.0000, 0.0000],
        [0.5403, 0.8415, 0.9999, 0.0100]])

In [654]:
import torch

def softmax(x: torch.Tensor, dim: int) -> torch.Tensor:
    x_max = x.max(dim=-1, keepdim=True).values
    x_norm = x.subtract(x_max)
    return x_norm.exp() / x_norm.exp().sum(dim=dim, keepdim=True)

In [655]:
x = torch.randn(2, 3)
print(x)
y = x.max(dim=-1, keepdim=True).values
print(y)
print(x.subtract(y))
print(softmax(x, -1))

tensor([[ 1.3352, -2.0181, -0.6055],
        [-2.0257, -0.4913,  0.5074]])
tensor([[1.3352],
        [0.5074]])
tensor([[ 0.0000, -3.3533, -1.9407],
        [-2.5331, -0.9987,  0.0000]])
tensor([[0.8485, 0.0297, 0.1218],
        [0.0548, 0.2544, 0.6907]])


In [656]:
torch.allclose(torch.softmax(x, dim=-1), softmax(x, -1))

True

In [657]:
import torch 
from jaxtyping import Bool, Float

d_k = 4
d_v = 3
keys = 2
queries = 2

Q: Float[torch.Tensor, " ... queries d_k"] = torch.randn(keys, d_k)
K: Float[torch.Tensor, " ... keys d_k"] = torch.randn(queries, d_k)
V: Float[torch.Tensor, " ... keys d_v"] = torch.randn(keys, d_v)
mask: Bool[torch.Tensor, " ... queries keys"] | None = (torch.randn(queries, keys).uniform_() > 0.8)

In [658]:
import torch
from jaxtyping import Bool, Float
from einops import einsum
import math

def scaled_dot_product_attention(
        Q: Float[torch.Tensor, " ... queries d_k"],
        K: Float[torch.Tensor, " ... keys d_k"],
        V: Float[torch.Tensor, " ... keys d_v"],
        mask: Bool[torch.Tensor, " ... queries keys"] | None = None
) -> Float[torch.Tensor, " ... seq_len d_v"]:
    d_k = Q.shape[-1]
    qk = einsum(Q, K, "... queries d_k, ... keys d_k -> ... queries keys") / math.sqrt(d_k)
    if mask is not None:
        qk = qk.masked_fill(~mask, float("-Inf"))
    qk = softmax(qk, dim=-1)
    return einsum(qk, V, "... queries keys, ... keys d_v -> ... queries d_v")

In [659]:
scaled_dot_product_attention(Q, K, V, mask)

tensor([[-1.2173,  0.1334, -0.7420],
        [    nan,     nan,     nan]])

In [660]:
torch.allclose(torch.nn.functional.scaled_dot_product_attention(Q, K, V), scaled_dot_product_attention(Q, K, V))

True

In [663]:
import torch
from einops import einsum, rearrange

class CasualMultiHeadSelfAttention(torch.nn.Module):

    device: torch.device | None
    d_k: int
    d_v: int
    d_model: int
    num_heads: int
    weights_q: Linear # (num_heads * d_k, d_model) = (hd_k, d_model)
    weights_k: Linear # (num_heads * d_k, d_model) = (hd_k, d_model)
    weights_v: Linear # (num_heads * d_v, d_model) = (hd_v, d_model)
    weights_o: Linear # (d_model, num_heads * d_v) = (d_model, hd_v)
    rope: RotaryPositionalEmbedding | None

    def __init__(self, d_model: int, num_heads: int, max_seq_len: int | None = None, theta: float | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        if d_model % num_heads != 0:
            raise Exception("d_model must be divisible by num_heads!")
        self.device = device
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = round(d_model / num_heads)
        self.d_v = self.d_k
        self.weights_q = Linear(d_model, num_heads * self.d_k, device, dtype)
        self.weights_k = Linear(d_model, num_heads * self.d_k, device, dtype)
        self.weights_v = Linear(d_model, num_heads * self.d_v, device, dtype)
        self.weights_o = Linear(num_heads * self.d_v, d_model, device, dtype)
        if max_seq_len is not None and theta is not None:
            self.rope = RotaryPositionalEmbedding(theta, self.d_k, max_seq_len)
        else:
            self.rope = None

    # x (..., seq_len, d_model)
    def forward(self, x: torch.Tensor, token_positions: torch.Tensor | None = None) -> torch.Tensor:
        seq_len = x.shape[-2]
        casual_mask = (torch.tril(torch.ones(seq_len, seq_len, device=self.device)) == 1)
        if token_positions is None:
            token_positions = torch.arange(seq_len, device=self.device)
        wq_x = self.weights_q(x)  # (..., seq_len, hd_k)
        wk_x = self.weights_k(x)  # (..., seq_len, hd_k)
        wv_x = self.weights_v(x)  # (..., seq_len, hd_v)
        wq_x_i = rearrange(wq_x, "... seq_len (h d_k) -> ... h seq_len d_k", h=self.num_heads)  # (..., h, seq_len, d_k)
        wk_x_i = rearrange(wk_x, "... seq_len (h d_k) -> ... h seq_len d_k", h=self.num_heads)  # (..., h, seq_len, d_k)
        wv_x_i = rearrange(wv_x, "... seq_len (h d_v) -> ... h seq_len d_v", h=self.num_heads)  # (..., h, seq_len, d_v)
        if self.rope:
            wq_x_i = self.rope(wq_x_i, token_positions)
            wk_x_i = self.rope(wk_x_i, token_positions)
        result = scaled_dot_product_attention(wq_x_i, wk_x_i, wv_x_i, casual_mask)  # (..., h, seq_len, d_v)
        result = rearrange(result, "... h seq_len d_v -> ... seq_len (h d_v)")
        return self.weights_o(result)


In [664]:
import torch
from einops import einsum, rearrange

class CasualMultiHeadSelfAttentionOptimized(torch.nn.Module):

    device: torch.device | None
    d_k: int
    d_v: int
    d_model: int
    num_heads: int
    weights_qkv: Linear # (hd_k + hd_k + hd_v, d_model)
    weights_o: Linear # (d_model, num_heads * d_v) = (d_model, hd_v)
    rope: RotaryPositionalEmbedding | None

    def __init__(self, d_model: int, num_heads: int, max_seq_len: int | None = None, theta: float | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        if d_model % num_heads != 0:
            raise Exception("d_model must be divisible by num_heads!")
        self.device = device
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = round(d_model / num_heads)
        self.d_v = self.d_k
        self.weights_qkv = Linear(d_model, num_heads * self.d_k + num_heads * self.d_k + num_heads * self.d_v, device, dtype)
        self.weights_o = Linear(num_heads * self.d_v, d_model, device, dtype)
        if max_seq_len is not None and theta is not None:
            self.rope = RotaryPositionalEmbedding(theta, self.d_k, max_seq_len)
        else:
            self.rope = None

    # x (..., seq_len, d_model)
    def forward(self, x: torch.Tensor, token_positions: torch.Tensor | None = None) -> torch.Tensor:
        seq_len = x.shape[-2]
        casual_mask = (torch.tril(torch.ones(seq_len, seq_len, device=self.device)) == 1)
        if token_positions is None:
            token_positions = torch.arange(seq_len, device=self.device)
        w_x = self.weights_qkv(x)  # (..., seq_len, hd_k + hd_k + hd_v)
        wq_x = w_x[..., :self.num_heads * self.d_k]
        wk_x = w_x[..., self.num_heads * self.d_k:2 * self.num_heads * self.d_k]
        wv_x = w_x[..., 2 * self.num_heads * self.d_k:]
        wq_x_i = rearrange(wq_x, "... seq_len (h d_k) -> ... h seq_len d_k", h=self.num_heads)  # (..., h, seq_len, d_k)
        wk_x_i = rearrange(wk_x, "... seq_len (h d_k) -> ... h seq_len d_k", h=self.num_heads)  # (..., h, seq_len, d_k)
        wv_x_i = rearrange(wv_x, "... seq_len (h d_v) -> ... h seq_len d_v", h=self.num_heads)  # (..., h, seq_len, d_v)
        if self.rope:
            wq_x_i = self.rope(wq_x_i, token_positions)
            wk_x_i = self.rope(wk_x_i, token_positions)
        result = scaled_dot_product_attention(wq_x_i, wk_x_i, wv_x_i, casual_mask)  # (..., h, seq_len, d_v)
        result = rearrange(result, "... h seq_len d_v -> ... seq_len (h d_v)")
        return self.weights_o(result)


In [673]:
d_model = 16
num_heads = 4
max_seq_len = 8
theta = 10000
attention = CasualMultiHeadSelfAttention(d_model, num_heads)
print(attention)
attention_with_rope = CasualMultiHeadSelfAttention(d_model, num_heads, max_seq_len, theta)
attention_with_rope_optimized = CasualMultiHeadSelfAttentionOptimized(d_model, num_heads, max_seq_len, theta)
print(attention_with_rope)
x = torch.randn(max_seq_len, d_model)
print(attention(x))
print(attention_with_rope(x))
print(attention_with_rope_optimized(x))

CasualMultiHeadSelfAttention(
  (weights_q): Linear()
  (weights_k): Linear()
  (weights_v): Linear()
  (weights_o): Linear()
)
CasualMultiHeadSelfAttention(
  (weights_q): Linear()
  (weights_k): Linear()
  (weights_v): Linear()
  (weights_o): Linear()
  (rope): RotaryPositionalEmbedding()
)
tensor([[-1.0325, -0.6535,  0.8761,  0.6417,  0.4070,  0.1905,  0.7393, -1.4914,
         -0.1519,  0.1578, -1.7584,  1.9562,  0.1731,  0.3230,  1.0046,  1.0474],
        [-0.0543, -0.3147, -0.0914,  0.2874, -0.1873,  0.0346, -0.3483, -0.6583,
          0.1231,  0.0224, -0.8080,  0.0162, -0.0210,  0.0877,  0.3362, -0.1018],
        [ 0.4703, -0.0575, -0.1600, -0.2685, -0.2014,  0.4721, -1.0844,  0.2323,
         -0.0218,  0.1327, -0.2337, -0.6228, -0.5138,  0.7501, -0.7166, -0.4392],
        [-0.0871, -0.4068,  0.5270, -0.1126,  0.0368, -0.1099, -0.0662, -0.2440,
         -0.3771, -0.1598, -0.5155,  0.9245, -0.4246,  0.4674,  0.0253,  0.2663],
        [ 0.2996,  0.2976,  0.2272,  0.1687, -0.4571, 

In [666]:
import torch

class TransformerBlock(torch.nn.Module):

    norm_attention: RMSNorm
    attention: CasualMultiHeadSelfAttention

    feed_forward: SwiGLU
    norm_feed_forward: RMSNorm

    def __init__(self, d_model: int, num_heads: int, d_ff: int | None = None, eps: float = 1e-5, max_seq_len: int | None = None, theta: float | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.norm_attention = RMSNorm(d_model, eps, device=device, dtype=dtype)
        self.attention = CasualMultiHeadSelfAttention(d_model, num_heads, max_seq_len, theta, device=device, dtype=dtype)
        self.norm_feed_forward = RMSNorm(d_model, eps, device=device, dtype=dtype)
        self.feed_forward = SwiGLU(d_model, d_ff, device=device, dtype=dtype)

    def forward(self, x: torch.Tensor, token_positions: torch.Tensor | None = None) -> torch.Tensor:
        y = x + self.attention(self.norm_attention(x), token_positions)
        y = y + self.feed_forward(self.norm_feed_forward(y))
        return y

In [667]:
d_model = 16
num_heads = 4
d_ff = round(8 * d_model / 3 / 64) * 64
transformer = TransformerBlock(d_model, num_heads, d_ff)
print(transformer)
x = torch.randn(4, d_model)
transformer(x)

TransformerBlock(
  (norm_attention): RMSNorm()
  (attention): CasualMultiHeadSelfAttention(
    (weights_q): Linear()
    (weights_k): Linear()
    (weights_v): Linear()
    (weights_o): Linear()
  )
  (norm_feed_forward): RMSNorm()
  (feed_forward): SwiGLU(
    (w_1): Linear()
    (w_2): Linear()
    (w_3): Linear()
  )
)


tensor([[-0.8361,  3.4925,  0.6838,  0.4776,  1.1359, -1.9560, -3.2085,  1.7651,
          2.7229,  1.4522,  1.1444,  0.6724, -0.0316, -1.5649, -0.1822,  0.4457],
        [-0.6846,  0.1729,  2.4477,  1.5535, -0.1492,  0.6471, -1.6785,  0.2343,
         -0.8890,  1.3885,  1.0085, -0.3680,  0.7970, -2.0652, -0.2940, -0.7685],
        [-0.4037,  1.6604, -2.2988,  2.1710, -0.0290,  1.0153, -0.9589, -1.0617,
         -1.0401,  2.5323,  1.6276, -0.5337, -1.4662, -2.0814,  1.7576,  0.3118],
        [-1.6067,  2.1019,  1.2130,  0.9095, -1.2566, -0.5850, -0.1243, -0.1286,
          1.2594, -0.6429,  0.0579,  1.6215,  1.2097, -0.1524,  1.6599, -0.6119]],
       grad_fn=<AddBackward0>)

In [668]:
import torch

class TransformerLM(torch.nn.Module):

    token_embeddings: Embedding
    layers: torch.nn.ModuleList
    ln_final: RMSNorm
    lm_head: Linear

    def __init__(self, vocab_size: int, context_length: int, num_layers: int, d_model: int, num_heads: int, d_ff: int | None = None, eps: float = 1e-5, theta: float | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.token_embeddings = Embedding(num_embeddings=vocab_size, embedding_dim=d_model, device=device, dtype=dtype)
        self.layers = torch.nn.ModuleList(TransformerBlock(d_model, num_heads, d_ff, eps, context_length, theta, device, dtype) for _ in range(num_layers))
        self.ln_final = RMSNorm(d_model, eps, device=device, dtype=dtype)
        self.lm_head = Linear(d_model, vocab_size, device=device, dtype=dtype)
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        print(x)
        print(x.dtype)
        y = self.token_embeddings(x)
        for transformer in self.layers:
            y = transformer(y)
        y = self.ln_final(y)
        y = self.lm_head(y)
        y = softmax(y, -1)
        return y

In [669]:
vocab_size = 256
context_length = 1000
num_layers = 2
d_model = 16
num_heads = 4
d_ff = round(8 * d_model / 3 / 64) * 64
eps: float = 1e-5
theta = 10000

transformer_lm = TransformerLM(vocab_size, context_length, num_layers, d_model, num_heads, d_ff, eps, theta)
print(transformer_lm)

x = torch.randint(0, 255, (4, 6))
y = transformer_lm(x)
print(y)

TransformerLM(
  (token_embeddings): Embedding()
  (layers): ModuleList(
    (0-1): 2 x TransformerBlock(
      (norm_attention): RMSNorm()
      (attention): CasualMultiHeadSelfAttention(
        (weights_q): Linear()
        (weights_k): Linear()
        (weights_v): Linear()
        (weights_o): Linear()
        (rope): RotaryPositionalEmbedding()
      )
      (norm_feed_forward): RMSNorm()
      (feed_forward): SwiGLU(
        (w_1): Linear()
        (w_2): Linear()
        (w_3): Linear()
      )
    )
  )
  (ln_final): RMSNorm()
  (lm_head): Linear()
)
tensor([[231,  94, 175, 165,  88, 218],
        [ 64, 239,  27, 214, 118, 193],
        [ 41,  18, 239, 157, 203, 119],
        [ 60,  50, 232,  31,  87,  26]])
torch.int64
tensor([[[0.0041, 0.0033, 0.0031,  ..., 0.0045, 0.0045, 0.0031],
         [0.0049, 0.0030, 0.0039,  ..., 0.0052, 0.0026, 0.0022],
         [0.0057, 0.0025, 0.0029,  ..., 0.0038, 0.0031, 0.0031],
         [0.0032, 0.0033, 0.0028,  ..., 0.0036, 0.0032, 0.0041],
 

In [670]:
def print_model_summary(model, input_size):
    def register_hook(module):
        def hook(module, input, output):
            class_name = str(module.__class__).split(".")[-1].split("'")[0]
            module_idx = len(summary)
            m_key = f"{class_name}-{module_idx+1}"
            summary[m_key] = {
                "input_shape": list(input[0].size()),
                "output_shape": list(output.size()),
                "nb_params": sum(p.numel() for p in module.parameters(recurse=False))
            }
        if not isinstance(module, nn.Sequential) and not isinstance(module, nn.ModuleList) and module != model:
            hooks.append(module.register_forward_hook(hook))

    summary = {}
    hooks = []
    model.apply(register_hook)
    with torch.no_grad():
        model(torch.zeros(1, *input_size, dtype=torch.long))

    for h in hooks:
        h.remove()

    print("----------------------------------------------------------------")
    line_new = "{:>20}  {:>25} {:>15}".format("Layer (type)", "Output Shape", "Param #")
    print(line_new)
    print("================================================================")
    total_params = 0
    for layer in summary:
        line_new = "{:>20}  {:>25} {:>15}".format(
            layer,
            str(summary[layer]["output_shape"]),
            "{0:,}".format(summary[layer]["nb_params"])
        )
        total_params += summary[layer]["nb_params"]
        print(line_new)
    print("================================================================")
    print(f"Total params: {total_params:,}")
    print("----------------------------------------------------------------")

# Example usage
print_model_summary(transformer_lm, (context_length, ))

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0

In [671]:
total = sum(p.numel() for p in transformer_lm.parameters())
total

16464